# Inference Layer - Complete Feature Showcase

This notebook walks through the **entire public API** of the low-level `inference` package. Every exported function and type is used below.

## 1. Setup & Imports

In [7]:
from pathlib import Path

# Import the full public API so every symbol is demonstrated
from inference import (
    InferenceRequest,
    InferenceResult,
    create_client,
    run_batch,
    run_completion,
)


# Resolve config path so the notebook works from repo root or from examples/
def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "inference.example.yaml").exists():
            return p
    return Path.cwd()


CONFIG_PATH = _repo_root() / "config" / "inference.example.yaml"

## 2. Configuration and `InferenceConfig`

Configuration is loaded from a YAML file. The structure is described by the **`InferenceConfig`** type (providers, model_aliases, rate_limit, retry, log_path, checkpoint_path). You do not construct `InferenceConfig` by hand; you pass a path to `create_client()` or `run_batch()`, and the package loads and validates the file into an `InferenceConfig` internally. Per-provider **concurrency** (how many requests in flight at once) is set under each provider: `max_concurrency` and optionally `per_model_concurrency`; use 0 for unlimited. See `config/inference.example.yaml` for the full schema.

In [8]:
# Path to the example config (works from repo root or examples/)
config_path = CONFIG_PATH

## 3. Single Completion: `create_client`, `run_completion`, `InferenceRequest`, `InferenceResult`

- **`create_client(config_path)`** returns a client used for inference.
- **`InferenceRequest`** is built with `model_alias`, `prompt`, and optionally `max_tokens`, `temperature`.
- **`run_completion(client, request)`** runs one request and returns an **`InferenceResult`**, which has: `model_alias`, `provider`, `model`, `content`, `prompt_tokens`, `completion_tokens`, `total_tokens`, `latency_ms`, `retry_count`.

In [3]:
# create_client loads the YAML and builds the client
client = create_client(CONFIG_PATH)

# InferenceRequest: model_alias, prompt, max_tokens (optional), temperature (optional)
request = InferenceRequest(
    model_alias="gemma-3-4b",
    prompt="What is the capital of France?",
    max_tokens=100,
    temperature=0.7,
)

# run_completion returns InferenceResult
result: InferenceResult = await run_completion(client, request)

# InferenceResult fields (all part of the public API)
print("Content:", result.content)
print("Provider:", result.provider, "| Model:", result.model)
print(
    "Tokens: prompt",
    result.prompt_tokens,
    "+ completion",
    result.completion_tokens,
    "(total",
    result.total_tokens,
    ")",
)
print("Latency:", f"{result.latency_ms:.0f} ms", "| Retries:", result.retry_count)

Content: The capital of France is **Paris**. 

It’s a global center for art, fashion, gastronomy and culture. 😊 

Would you like to know anything more about Paris?
Provider: openrouter | Model: google/gemma-3-4b-it:free
Tokens: prompt 8 + completion 0 (total 8 )
Latency: 2271 ms | Retries: 0


## 4. Batch Processing: `run_batch`

**`run_batch(config_path, requests)`** takes a config path and an async iterator of **`InferenceRequest`**s. It creates the client and a batch runner internally, writes progress to checkpoints (so you can resume after interruption), and logs metadata to the configured log file (e.g. `logs/inference.jsonl`). Checkpoints go to e.g. `checkpoints/batch.jsonl`.

In [4]:
async def generate_batch_requests():
    """Yield InferenceRequest objects; run_batch handles checkpointing and logging."""
    prompts = [
        "What is 2+2?",
        "What is the capital of Germany?",
        "Name a primary color.",
    ]
    for prompt in prompts:
        yield InferenceRequest(model_alias="gemma-3-4b", prompt=prompt, max_tokens=50)


await run_batch(CONFIG_PATH, generate_batch_requests())
# Outputs: logs/inference.jsonl (metadata only), checkpoints/batch.jsonl (resume state)

## 5. Error Handling

Invalid `model_alias` (e.g. not present in config) raises **`UnknownModelAliasError`**. Failed requests after retries raise **`InferenceRequestError`**. Use try/except when you need to handle these.

In [5]:
client = create_client(CONFIG_PATH)

# Invalid model_alias raises (e.g. UnknownModelAliasError); catch and handle as needed
try:
    await run_completion(client, InferenceRequest(model_alias="nonexistent-model", prompt="Hello"))
except Exception as e:
    print("Error (invalid alias):", type(e).__name__, "-", e)

# Valid request
result = await run_completion(client, InferenceRequest(model_alias="gemma-3-4b", prompt="Hello"))
print("Success:", result.content[:60], "...")

Error (invalid alias): UnknownModelAliasError - Unknown model alias 'nonexistent-model'. Configured aliases: ['gemma-3-4b', 'llama-3.2-3b', 'mistral-small-24b', 'mock-test', 'qwen3-4b', 'step-3.5-flash']
Success: Hello there! How can I help you today? 😊 

Do you want to:

 ...


## 6. Log Analysis

After `run_batch`, the log file (e.g. `logs/inference.jsonl`) contains one JSON line per request: status, provider, model, tokens, latency, and error info. It does **not** store prompt or response content. The checkpoint file lists completed requests so the runner can skip them on resume.

In [6]:
import json

log_path = Path("logs/inference.jsonl")
if log_path.exists():
    entries = [json.loads(line) for line in open(log_path) if line.strip()]
    success = sum(1 for e in entries if e.get("status") == "success")
    failures = sum(1 for e in entries if e.get("status") == "failure")
    print("Log entries:", len(entries), "| Success:", success, "| Failures:", failures)
    if entries:
        for entry in reversed(entries):
            if entry.get("status") == "success":
                print(json.dumps(entry, indent=2))
                break
else:
    print("No log file at", log_path)

Log entries: 52 | Success: 52 | Failures: 0
{
  "provider": "openrouter",
  "model": "google/gemma-3-4b-it:free",
  "status": "success",
  "latency_ms": 2030.1372287794948,
  "timestamp": "2026-03-12T20:41:35.434449Z",
  "prompt_tokens": 2,
  "completion_tokens": 0,
  "total_tokens": 2
}


## Summary – All Public API Symbols

| Symbol | Type | Purpose |
|--------|------|---------|
| `create_client` | function | Build client from config path (YAML → InferenceConfig internally) |
| `run_completion` | async function | Run one request; returns `InferenceResult` |
| `run_batch` | async function | Run async iterator of `InferenceRequest` with checkpointing |
| `InferenceRequest` | dataclass | `model_alias`, `prompt`, `max_tokens`, `temperature` |
| `InferenceResult` | dataclass | `content`, `provider`, `model`, token counts, `latency_ms`, `retry_count` |
| `InferenceConfig` | type | Schema for the YAML config (used internally) |

**Next:** `experiments_example.ipynb` for the high-level experiments layer (matrix runs, resume, scheduling).